### PHẦN 3: MỐI QUAN HỆ VÀ PLOTLY NÂNG CAO

Phần này tập trung phân tích mối quan hệ giữa các chỉ tiêu du lịch và khai thác khả năng trực quan hóa tương tác của Plotly.

### Nội dung thực hiện

- Phân tích mối quan hệ giữa lượng khách và doanh thu bằng Scatter Plot.
- Phân tích tương quan giữa các chỉ tiêu du lịch bằng Correlation Heatmap.
- Trực quan hóa sự thay đổi thứ hạng doanh thu địa phương qua các năm bằng Animated Bar Chart.
- Tổng hợp các insight chính của Chương 3.

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

PATH_NAM = Path("../data/processed/du_lieu_du_lich_nam_clean.csv")
PATH_DIA = Path("../data/processed/du_lieu_du_lich_dia_phuong_clean.csv")

df_nam = pd.read_csv(PATH_NAM, encoding="utf-8-sig")
df_dia = pd.read_csv(PATH_DIA, encoding="utf-8-sig")

print("Dữ liệu theo năm:", df_nam.shape)
print("Dữ liệu địa phương:", df_dia.shape)

Dữ liệu theo năm: (10, 13)
Dữ liệu địa phương: (700, 5)


In [23]:
# Kiểm tra các cột còn giá trị thiếu
missing_nam = df_nam.isna().sum()
missing_nam = missing_nam[missing_nam > 0]

print("CÁC CỘT CÒN GIÁ TRỊ THIẾU")
display(
    missing_nam.to_frame("Số giá trị thiếu")
)

# Kiểm tra cấp dữ liệu địa phương
print("\nCẤP DỮ LIỆU ĐỊA PHƯƠNG")
display(
    df_dia["Cấp dữ liệu"]
          .value_counts()
          .to_frame("Số quan sát")
)

CÁC CỘT CÒN GIÁ TRỊ THIẾU


,Số giá trị thiếu
Khách nghỉ qua đêm (Nghìn lượt),6
Khách trong ngày (Nghìn lượt),6



CẤP DỮ LIỆU ĐỊA PHƯƠNG


,Số quan sát
Cấp dữ liệu,
Tỉnh/thành,630
Vùng,60
Cả nước,10


### Nhận xét dữ liệu đầu vào

- Bộ dữ liệu theo năm có 13 thuộc tính, trong đó phần lớn các chỉ tiêu phục vụ phân tích đều có đầy đủ dữ liệu trong giai đoạn 2015–2024.
- Hai biến `Khách nghỉ qua đêm` và `Khách trong ngày` cùng thiếu 6 giá trị, vì vậy không được lựa chọn làm biến chính trong các phân tích tương quan của TV4.
- Bộ dữ liệu địa phương gồm ba cấp dữ liệu: `Cả nước`, `Vùng` và `Tỉnh/thành`.
- Trong 700 quan sát của dữ liệu địa phương, có 630 quan sát thuộc cấp `Tỉnh/thành`, tương ứng với dữ liệu của 63 tỉnh/thành qua 10 năm.
- Khi xây dựng bảng xếp hạng và Animated Bar Chart, chỉ dữ liệu cấp `Tỉnh/thành` được sử dụng để tránh trộn lẫn các cấp tổng hợp khác nhau.

## 3.7. Mối quan hệ giữa lượng khách và doanh thu

### Biểu đồ 7 – Scatter Plot: Lượng khách và doanh thu cơ sở lưu trú

**Câu hỏi phân tích:**  
Trong giai đoạn 2015–2024, lượng khách cơ sở lưu trú phục vụ và doanh thu cơ sở lưu trú có xu hướng biến động cùng chiều hay không?

**Biến sử dụng:**

- Trục X: Khách cơ sở lưu trú phục vụ (Nghìn lượt).
- Trục Y: Doanh thu cơ sở lưu trú (Tỷ đồng).
- Thông tin tương tác: Năm.

**Mục đích:**  
Scatter Plot được sử dụng để quan sát mối quan hệ giữa hai biến định lượng. Hệ số tương quan Pearson được tính thêm nhằm hỗ trợ đánh giá mức độ liên hệ tuyến tính giữa lượng khách và doanh thu.

> Mối tương quan giữa hai biến không đồng nghĩa với quan hệ nhân quả.

In [24]:
# Chọn hai chỉ tiêu phân tích
col_khach = "Khách cơ sở lưu trú phục vụ (Nghìn lượt)"
col_doanh_thu = "Doanh thu cơ sở lưu trú (Tỷ đồng)"

scatter_data = df_nam[
    ["Năm", col_khach, col_doanh_thu]
].dropna()

# Hệ số tương quan Pearson
pearson_r = scatter_data[col_khach].corr(
    scatter_data[col_doanh_thu]
)

print(f"Hệ số tương quan Pearson: r = {pearson_r:.3f}")

# Vẽ Scatter Plot
fig7 = px.scatter(
    scatter_data,
    x=col_khach,
    y=col_doanh_thu,
    text="Năm",
    title="Mối quan hệ giữa lượng khách và doanh thu lưu trú"
)

fig7.update_traces(
    marker=dict(size=11),
    textposition="top center",
    hovertemplate=(
        "Năm: %{text}<br>"
        "Khách: %{x:,.0f} nghìn lượt<br>"
        "Doanh thu: %{y:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

fig7.update_layout(
    xaxis_title="Khách cơ sở lưu trú phục vụ (Nghìn lượt)",
    yaxis_title="Doanh thu cơ sở lưu trú (Tỷ đồng)"
)

fig7.show()

Hệ số tương quan Pearson: r = 0.994


### Nhận xét Biểu đồ 7

- Lượng khách và doanh thu cơ sở lưu trú có xu hướng tăng, giảm cùng chiều.
- Hệ số **Pearson r = 0.994** cho thấy tương quan tuyến tính dương rất mạnh giữa hai chỉ tiêu.
- Năm 2021 ở mức thấp nhất, trong khi năm 2024 đạt mức cao nhất về cả lượng khách và doanh thu.

## 3.8. Tương quan giữa các chỉ tiêu du lịch

### Biểu đồ 8 – Correlation Heatmap

**Câu hỏi phân tích:**  
Các chỉ tiêu về doanh thu và lượng khách du lịch có mức độ tương quan với nhau như thế nào trong giai đoạn 2015–2024?

**Các biến được lựa chọn:**

- Doanh thu cơ sở lưu trú.
- Doanh thu cơ sở lữ hành.
- Khách cơ sở lưu trú phục vụ.
- Khách trong nước tại cơ sở lưu trú.
- Khách quốc tế tại cơ sở lưu trú.
- Khách cơ sở lữ hành phục vụ.

**Loại biểu đồ:** Correlation Heatmap.

**Mục đích:**  
Ma trận tương quan giúp đánh giá đồng thời mức độ liên hệ tuyến tính giữa nhiều chỉ tiêu du lịch. Heatmap được sử dụng để trực quan hóa các hệ số tương quan, từ đó xác định những cặp biến có mối liên hệ mạnh hoặc yếu và hỗ trợ kết quả phân tích Scatter Plot ở mục 3.7.

> Hệ số tương quan phản ánh mức độ liên hệ giữa các biến nhưng không chứng minh quan hệ nhân quả.

In [16]:
# Các biến sử dụng cho phân tích tương quan
corr_cols = [
    "Doanh thu cơ sở lưu trú (Tỷ đồng)",
    "Doanh thu cơ sở lữ hành (Tỷ đồng)",
    "Khách cơ sở lưu trú phục vụ (Nghìn lượt)",
    "Khách trong nước - cơ sở lưu trú (Nghìn lượt)",
    "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)",
    "Khách cơ sở lữ hành phục vụ (Nghìn lượt)"
]

# Tạo dữ liệu phân tích
corr_data = df_nam[corr_cols].copy()

# Tính ma trận tương quan Pearson
corr_matrix = corr_data.corr(method="pearson")

print("MA TRẬN TƯƠNG QUAN PEARSON")
display(corr_matrix.round(3))


# Đổi tên ngắn gọn để biểu đồ dễ đọc
short_names = {
    "Doanh thu cơ sở lưu trú (Tỷ đồng)": "DT lưu trú",
    "Doanh thu cơ sở lữ hành (Tỷ đồng)": "DT lữ hành",
    "Khách cơ sở lưu trú phục vụ (Nghìn lượt)": "Khách lưu trú",
    "Khách trong nước - cơ sở lưu trú (Nghìn lượt)": "Khách nội địa",
    "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)": "Khách quốc tế",
    "Khách cơ sở lữ hành phục vụ (Nghìn lượt)": "Khách lữ hành"
}

corr_display = corr_matrix.rename(
    index=short_names,
    columns=short_names
)

# Vẽ Heatmap
fig8 = px.imshow(
    corr_display,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Ma trận tương quan giữa các chỉ tiêu du lịch"
)

fig8.update_layout(
    xaxis_title="Chỉ tiêu",
    yaxis_title="Chỉ tiêu"
)

fig8.show()

MA TRẬN TƯƠNG QUAN PEARSON


,Doanh thu cơ sở lưu trú (Tỷ đồng),Doanh thu cơ sở lữ hành (Tỷ đồng),Khách cơ sở lưu trú phục vụ (Nghìn lượt),Khách trong nước - cơ sở lưu trú (Nghìn lượt),Khách quốc tế - cơ sở lưu trú (Nghìn lượt),Khách cơ sở lữ hành phục vụ (Nghìn lượt)
Doanh thu cơ sở lưu trú (Tỷ đồng),1.000,0.987,0.994,0.973,0.913,0.982
Doanh thu cơ sở lữ hành (Tỷ đồng),0.987,1.000,0.991,0.959,0.936,0.989
Khách cơ sở lưu trú phục vụ (Nghìn lượt),0.994,0.991,1.000,0.985,0.905,0.979
Khách trong nước - cơ sở lưu trú (Nghìn lượt),0.973,0.959,0.985,1.000,0.816,0.933
Khách quốc tế - cơ sở lưu trú (Nghìn lượt),0.913,0.936,0.905,0.816,1.000,0.962
Khách cơ sở lữ hành phục vụ (Nghìn lượt),0.982,0.989,0.979,0.933,0.962,1.000


In [17]:
from itertools import combinations

pair_results = []

for col1, col2 in combinations(corr_cols, 2):
    r = corr_matrix.loc[col1, col2]

    pair_results.append({
        "Biến 1": short_names[col1],
        "Biến 2": short_names[col2],
        "r": r
    })

pair_corr = pd.DataFrame(pair_results)

# Cặp có |r| lớn nhất
strongest = pair_corr.loc[pair_corr["r"].abs().idxmax()]

# Cặp có |r| nhỏ nhất
weakest = pair_corr.loc[pair_corr["r"].abs().idxmin()]

print(
    f"Cặp tương quan mạnh nhất: "
    f"{strongest['Biến 1']} ↔ {strongest['Biến 2']} "
    f"(r = {strongest['r']:.3f})"
)

print(
    f"Cặp tương quan yếu nhất: "
    f"{weakest['Biến 1']} ↔ {weakest['Biến 2']} "
    f"(r = {weakest['r']:.3f})"
)

Cặp tương quan mạnh nhất: DT lưu trú ↔ Khách lưu trú (r = 0.994)
Cặp tương quan yếu nhất: Khách nội địa ↔ Khách quốc tế (r = 0.816)


### Nhận xét Biểu đồ 8

- Correlation Heatmap cho thấy các chỉ tiêu về doanh thu và lượng khách du lịch trong giai đoạn 2015–2024 có mối tương quan dương tương đối mạnh với nhau.
- Cặp có mức tương quan mạnh nhất là **Doanh thu cơ sở lưu trú và Khách cơ sở lưu trú phục vụ**, với hệ số **r = 0.994**. Kết quả này phù hợp với Scatter Plot ở Biểu đồ 7, cho thấy khi lượng khách lưu trú ở mức cao hơn thì doanh thu cơ sở lưu trú cũng có xu hướng ở mức cao hơn.
- Cặp có hệ số tương quan thấp nhất trong nhóm biến được phân tích là **Khách trong nước tại cơ sở lưu trú và Khách quốc tế tại cơ sở lưu trú**, với **r = 0.816**. Tuy đây là giá trị thấp nhất trong ma trận, hệ số vẫn cho thấy mối tương quan dương mạnh.
- Như vậy, trong các biến được lựa chọn, không xuất hiện cặp có tương quan tuyến tính yếu hoặc tương quan âm; các chỉ tiêu nhìn chung có xu hướng biến động cùng chiều trong giai đoạn nghiên cứu.
- Heatmap giúp mở rộng kết quả của Scatter Plot từ một cặp biến sang nhiều chỉ tiêu du lịch, qua đó cho thấy mối liên hệ khá chặt chẽ giữa quy mô lượng khách và các chỉ tiêu doanh thu.
- Tuy nhiên, bộ dữ liệu chỉ gồm **10 quan sát theo năm**, do đó các hệ số tương quan cần được diễn giải trong phạm vi giai đoạn 2015–2024. Tương quan cao cũng **không chứng minh quan hệ nhân quả** giữa các chỉ tiêu.

## 3.9. Trực quan hóa tương tác nâng cao

### Biểu đồ 9 – Animated Bar Chart: Sự thay đổi thứ hạng doanh thu địa phương

**Câu hỏi phân tích:**  
Thứ hạng doanh thu du lịch lữ hành của các tỉnh/thành thay đổi như thế nào trong giai đoạn 2015–2024?

**Biến sử dụng:**

- Địa phương.
- Năm.
- Doanh thu du lịch lữ hành (Tỷ đồng).
- Cấp dữ liệu: `Tỉnh/thành`.

**Loại biểu đồ:** Animated Horizontal Bar Chart.

**Mục đích:**  
Biểu đồ động được sử dụng để theo dõi sự thay đổi thứ hạng doanh thu du lịch lữ hành của các tỉnh/thành qua từng năm. Khác với biểu đồ Top 10 tĩnh, Animated Bar Chart cho phép quan sát sự biến động thứ hạng trong toàn bộ giai đoạn 2015–2024 trên cùng một trực quan.

Việc sử dụng `animation_frame="Năm"` giúp thể hiện rõ khả năng tương tác và trực quan hóa dữ liệu theo thời gian của Plotly.

In [19]:
# Lọc riêng dữ liệu cấp Tỉnh/thành
df_tinh = df_dia[
    df_dia["Cấp dữ liệu"] == "Tỉnh/thành"
].copy()

# Loại các quan sát thiếu doanh thu
df_tinh = df_tinh.dropna(
    subset=["Doanh thu du lịch lữ hành (Tỷ đồng)"]
)

print("Số quan sát cấp Tỉnh/thành:", len(df_tinh))
print("Số tỉnh/thành:", df_tinh["Địa phương"].nunique())
print("Giai đoạn:", df_tinh["Năm"].min(), "-", df_tinh["Năm"].max())


# Lấy Top 10 tỉnh/thành có doanh thu cao nhất trong từng năm
top10_year = (
    df_tinh
    .sort_values(
        ["Năm", "Doanh thu du lịch lữ hành (Tỷ đồng)"],
        ascending=[True, False]
    )
    .groupby("Năm")
    .head(10)
    .copy()
)

# Tạo thứ hạng theo từng năm
top10_year["Hạng"] = (
    top10_year
    .groupby("Năm")["Doanh thu du lịch lữ hành (Tỷ đồng)"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Hiển thị Top 10 năm 2024 để kiểm tra
print("\nTOP 10 TỈNH/THÀNH NĂM 2024")

display(
    top10_year[
        top10_year["Năm"] == 2024
    ][
        [
            "Hạng",
            "Địa phương",
            "Doanh thu du lịch lữ hành (Tỷ đồng)"
        ]
    ].sort_values("Hạng")
)


# Sắp xếp dữ liệu cho animation
top10_animation = top10_year.sort_values(
    ["Năm", "Doanh thu du lịch lữ hành (Tỷ đồng)"],
    ascending=[True, True]
)

# Vẽ Animated Bar Chart
fig9 = px.bar(
    top10_animation,
    x="Doanh thu du lịch lữ hành (Tỷ đồng)",
    y="Địa phương",
    orientation="h",
    animation_frame="Năm",
    animation_group="Địa phương",
    text="Doanh thu du lịch lữ hành (Tỷ đồng)",
    hover_data={
        "Hạng": True,
        "Năm": True,
        "Doanh thu du lịch lữ hành (Tỷ đồng)": ":,.2f"
    },
    title="Top 10 tỉnh/thành theo doanh thu du lịch lữ hành qua các năm"
)

fig9.update_traces(
    texttemplate="%{x:,.1f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Doanh thu: %{x:,.2f} tỷ đồng"
        "<extra></extra>"
    )
)

# Giữ cùng thang đo cho toàn bộ animation
max_revenue = top10_animation[
    "Doanh thu du lịch lữ hành (Tỷ đồng)"
].max()

fig9.update_layout(
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Tỉnh/thành",
    xaxis=dict(
        range=[0, max_revenue * 1.12]
    ),
    showlegend=False
)

fig9.show()

Số quan sát cấp Tỉnh/thành: 618
Số tỉnh/thành: 63
Giai đoạn: 2015 - 2024

TOP 10 TỈNH/THÀNH NĂM 2024


,Hạng,Địa phương,Doanh thu du lịch lữ hành (Tỷ đồng)
559,1,TP. Hồ Chí Minh,31305.00
29,2,Hà Nội,25259.42
359,3,Đà Nẵng,5460.89
409,4,Khánh Hòa,4285.32
59,5,Quảng Ninh,1269.46
109,6,Hà Nam,991.51
389,7,Bình Định,942.48
79,8,Hải Phòng,795.27
659,9,Cần Thơ,764.09
649,10,Kiên Giang,761.04


### Nhận xét Biểu đồ 8

- Correlation Heatmap cho thấy các chỉ tiêu về **doanh thu và lượng khách du lịch** trong giai đoạn 2015–2024 nhìn chung có mối tương quan dương mạnh với nhau.
- Cặp có mức tương quan mạnh nhất là **Doanh thu cơ sở lưu trú và Khách cơ sở lưu trú phục vụ**, với hệ số **r = 0.994**. Kết quả này phù hợp với Scatter Plot ở Biểu đồ 7 và cho thấy hai chỉ tiêu có xu hướng biến động cùng chiều rất rõ.
- Cặp có hệ số tương quan thấp nhất trong các biến được lựa chọn là **Khách trong nước tại cơ sở lưu trú và Khách quốc tế tại cơ sở lưu trú**, với **r = 0.816**. Mặc dù đây là giá trị thấp nhất trong ma trận, mức tương quan này vẫn được xem là tương quan dương mạnh.
- Như vậy, trong nhóm chỉ tiêu được phân tích, không xuất hiện cặp biến có tương quan yếu hoặc tương quan âm. Các chỉ tiêu về lượng khách và doanh thu nhìn chung có xu hướng biến động cùng chiều trong giai đoạn nghiên cứu.
- Heatmap mở rộng kết quả của Scatter Plot từ một cặp biến sang nhiều chỉ tiêu, qua đó cho thấy mối liên hệ tương đối chặt chẽ giữa các hoạt động du lịch.
- Tuy nhiên, dữ liệu chỉ gồm **10 quan sát theo năm**, vì vậy kết quả cần được diễn giải trong phạm vi giai đoạn 2015–2024. **Tương quan cao không đồng nghĩa với quan hệ nhân quả.**

In [22]:
# =========================================================
# SO SÁNH THỨ HẠNG TOP 10 GIỮA NĂM 2015 VÀ 2024
# =========================================================

rank_2015 = (
    top10_year[top10_year["Năm"] == 2015]
    [["Địa phương", "Hạng", "Doanh thu du lịch lữ hành (Tỷ đồng)"]]
    .rename(columns={
        "Hạng": "Hạng 2015",
        "Doanh thu du lịch lữ hành (Tỷ đồng)": "Doanh thu 2015"
    })
)

rank_2024 = (
    top10_year[top10_year["Năm"] == 2024]
    [["Địa phương", "Hạng", "Doanh thu du lịch lữ hành (Tỷ đồng)"]]
    .rename(columns={
        "Hạng": "Hạng 2024",
        "Doanh thu du lịch lữ hành (Tỷ đồng)": "Doanh thu 2024"
    })
)

# Ghép Top 10 của hai năm
compare_rank = pd.merge(
    rank_2015,
    rank_2024,
    on="Địa phương",
    how="outer"
)

# Tính thay đổi thứ hạng đối với địa phương xuất hiện ở cả hai năm
compare_rank["Thay đổi hạng"] = (
    compare_rank["Hạng 2015"] - compare_rank["Hạng 2024"]
)

# Tính tốc độ thay đổi doanh thu nếu có dữ liệu ở cả hai năm
compare_rank["Tăng trưởng doanh thu (%)"] = (
    (compare_rank["Doanh thu 2024"] - compare_rank["Doanh thu 2015"])
    / compare_rank["Doanh thu 2015"]
    * 100
)

compare_rank = compare_rank.sort_values(
    ["Hạng 2024", "Hạng 2015"],
    na_position="last"
).reset_index(drop=True)

print("SO SÁNH TOP 10 NĂM 2015 VÀ 2024")

display(
    compare_rank.style.format({
        "Hạng 2015": "{:.0f}",
        "Doanh thu 2015": "{:,.2f}",
        "Hạng 2024": "{:.0f}",
        "Doanh thu 2024": "{:,.2f}",
        "Thay đổi hạng": "{:+.0f}",
        "Tăng trưởng doanh thu (%)": "{:+.2f}%"
    }, na_rep="—")
)

SO SÁNH TOP 10 NĂM 2015 VÀ 2024


,Địa phương,Hạng 2015,Doanh thu 2015,Hạng 2024,Doanh thu 2024,Thay đổi hạng,Tăng trưởng doanh thu (%)
0,TP. Hồ Chí Minh,1,"18,456.30",1,"31,305.00",+0,+69.62%
1,Hà Nội,2,"7,831.90",2,"25,259.42",+0,+222.52%
2,Đà Nẵng,3,"1,166.40",3,"5,460.89",+0,+368.18%
3,Khánh Hòa,7,197.40,4,"4,285.32",+3,+2070.88%
4,Quảng Ninh,4,434.80,5,"1,269.46",-1,+191.96%
5,Hà Nam,—,—,6,991.51,—,—
6,Bình Định,—,—,7,942.48,—,—
7,Hải Phòng,—,—,8,795.27,—,—
8,Cần Thơ,—,—,9,764.09,—,—
9,Kiên Giang,9,137.70,10,761.04,-1,+452.68%


### Nhận xét Biểu đồ 9

- Animated Bar Chart cho thấy thứ hạng doanh thu du lịch lữ hành giữa các tỉnh/thành có sự thay đổi đáng kể trong giai đoạn 2015–2024.

- **TP. Hồ Chí Minh** duy trì vị trí **hạng 1** ở cả năm 2015 và 2024. Doanh thu tăng từ **18.456,30 tỷ đồng** lên **31.305,00 tỷ đồng**, tương ứng mức tăng khoảng **69,62%**.

- **Hà Nội** tiếp tục giữ **hạng 2**, với doanh thu tăng từ **7.831,90 tỷ đồng** năm 2015 lên **25.259,42 tỷ đồng** năm 2024, tăng khoảng **222,52%**.

- **Đà Nẵng** cũng giữ nguyên **hạng 3**, trong khi doanh thu tăng từ **1.166,40 tỷ đồng** lên **5.460,89 tỷ đồng**, tương ứng mức tăng khoảng **368,18%**.

- **Khánh Hòa** có sự cải thiện thứ hạng nổi bật nhất trong nhóm các địa phương xuất hiện trong Top 10 ở cả hai năm, từ **hạng 7 năm 2015 lên hạng 4 năm 2024**, tăng **3 bậc**. Doanh thu tăng từ **197,40 tỷ đồng** lên **4.285,32 tỷ đồng**, tương ứng khoảng **2.070,88%**. Tuy nhiên, tỷ lệ tăng rất cao này một phần xuất phát từ mức doanh thu ban đầu tương đối thấp vào năm 2015.

- **Quảng Ninh** giảm từ **hạng 4 xuống hạng 5**, mặc dù doanh thu vẫn tăng từ **434,80 tỷ đồng** lên **1.269,46 tỷ đồng**. Tương tự, **Kiên Giang** giảm từ **hạng 9 xuống hạng 10**, dù doanh thu tăng từ **137,70 tỷ đồng** lên **761,04 tỷ đồng**.

- Nhóm Top 10 năm 2024 xuất hiện thêm bốn địa phương không có mặt trong Top 10 năm 2015 gồm **Hà Nam, Bình Định, Hải Phòng và Cần Thơ**.

- Ngược lại, bốn địa phương có mặt trong Top 10 năm 2015 nhưng không còn thuộc Top 10 năm 2024 gồm **Quảng Nam, Bà Rịa - Vũng Tàu, Quảng Bình và Huế**.

- Như vậy, có **6 địa phương xuất hiện trong Top 10 ở cả hai thời điểm 2015 và 2024**, gồm TP. Hồ Chí Minh, Hà Nội, Đà Nẵng, Khánh Hòa, Quảng Ninh và Kiên Giang. Điều này cho thấy nhóm dẫn đầu có một mức độ ổn định nhất định, đồng thời vẫn có sự thay đổi đáng kể ở các vị trí còn lại.

- Việc sử dụng `animation_frame="Năm"` giúp quan sát trực quan sự thay đổi cả về **quy mô doanh thu và thứ hạng địa phương** qua từng năm, thể hiện rõ ưu thế của Plotly so với biểu đồ Top 10 tĩnh.